# 03d CatBoost and categorical features

Optional, for after the session. Notebook 03 adds department, class and subclass to LightGBM as categorical features. This notebook asks two further questions with the scaled baseline of notebook 02:

- Does CatBoost, with its own handling of categorical features, do better than LightGBM on the same features?
- How do the two libraries expect categorical features: strings and `cat_features` for CatBoost, pandas `category` dtype for LightGBM?

It runs on its own, without notebook 03, in about eight minutes.

In [1]:
import os
import sys
import warnings

if "google.colab" in sys.modules:
    !git clone -q https://github.com/tackes/time-series-forecasting-bootcamp
    %cd time-series-forecasting-bootcamp
    %pip install -q statsforecast==2.1.1 mlforecast==1.1.0 utilsforecast==0.2.17 lightgbm==4.7.0 catboost==1.2.10
elif not os.path.exists("shared"):
    os.chdir("..")
sys.path.insert(0, os.getcwd())
warnings.filterwarnings("ignore")

In [2]:
import time

import numpy as np
import pandas as pd
import lightgbm as lgb
from mlforecast import MLForecast
from mlforecast.target_transforms import LocalStandardScaler

from shared.prepare_data import load_panel, H, FREQ, ALL_CUTOFFS, DEV_CUTOFFS, TEST_CUTOFF
from shared.features import LAGS, LAG_TRANSFORMS, DATE_FEATURES, LGB_PARAMS, CATEGORICAL, add_drivers, item_attributes
from shared.leaderboard import to_long, save_forecasts, load_forecasts, leaderboard

panel = load_panel()
panel = add_drivers(panel)                 # promo and rel_price, built step by step in notebook 03, section 4
actuals = panel[["unique_id", "ds", "y"]]
attributes = item_attributes(panel)        # department, class and subclass as strings

## The models

Three CatBoost candidates, each adding one thing to the one before:

| model | features |
|---|---|
| `catboost_baseline` | the 02 features: sales lags, rolling means, month and Fourier terms |
| `catboost_hist_exog` | plus three lagged drivers: promotion and realized relative price 14 weeks back, and the promotion share in the 26 weeks ending then |
| `catboost_exog_categorical` | plus department, class and subclass as native categorical features |

`lgbm_hist_exog` and `lgbm_exog_categorical` are LightGBM with the same features as the last two, with the attributes as pandas `category` columns so LightGBM splits on them as categories. Each feature set then exists for both algorithms, which separates what the features add from what the algorithm adds.

The LightGBM models here use the setup of notebook 03 (the 02 baseline). The attributes are fixed item properties, known at every origin. The hierarchy ids are numbers in the data but they are labels, so they enter as strings (CatBoost) or categories (LightGBM), never as numeric values. Brand is left out: 701 brands across 2,000 series is close to an item identifier.

CatBoost settings mirror `LGB_PARAMS` where the libraries have equivalents: 300 trees, learning rate 0.05, 80% of features per tree. The target scaler, lags and origins are the same as the LightGBM models. The categorical CatBoost model takes about 35 seconds per fit, so this section runs for about eight minutes.

The section shows the data at one cutoff, one CatBoost and one LightGBM fit, then runs all five models at every origin.

In [3]:
import time
from catboost import CatBoostRegressor

CB_PARAMS = dict(iterations=300, learning_rate=0.05, depth=6, rsm=0.8, random_seed=0, verbose=0)
hist_cols = ["promo_lag14", "rel_price_lag14", "promo_freq_26_lag14"]
cat_cols = CATEGORICAL

hist_features = panel[["unique_id", "ds"]].copy()
by_series = panel.groupby("unique_id")
hist_features["promo_lag14"] = by_series.promo.shift(H)
hist_features["rel_price_lag14"] = by_series.rel_price.shift(H)
hist_features["promo_freq_26_lag14"] = by_series.promo.transform(lambda s: s.shift(H).rolling(26, min_periods=1).mean())
hist_features = hist_features.fillna(-1)
hist_features[hist_features.promo_lag14 >= 0].head(3)

,unique_id,ds,promo_lag14,rel_price_lag14,promo_freq_26_lag14
14,10006_S1,2023-10-11,0.0,0.989145,0.0
15,10006_S1,2023-10-18,0.0,0.994472,0.0
16,10006_S1,2023-10-25,0.0,0.994807,0.0


### 1. The data at one cutoff

Three frames, built at the final cutoff:

- training data with the lagged drivers;
- the same with department, class and subclass added, as strings;
- the future frame `X_df`: the lagged drivers for the 14 forecast weeks. Each looks 14 weeks back, so every value is observed on the cutoff.

In [4]:
cutoff = TEST_CUTOFF
history = actuals[actuals.ds <= cutoff]

with_exog = history.merge(hist_features, on=["unique_id", "ds"])
with_categories = with_exog.merge(attributes, on="unique_id")
future_exog = hist_features[(hist_features.ds > cutoff) & (hist_features.ds <= cutoff + pd.Timedelta(weeks=H))]

display(with_exog.tail(3))
display(with_categories.tail(3))
future_exog.head(3)

,unique_id,ds,y,promo_lag14,rel_price_lag14,promo_freq_26_lag14
185297,9985_S1,2025-03-26,1106.0,0.0,0.919012,0.307692
185298,9985_S1,2025-04-02,1729.0,0.0,0.945223,0.307692
185299,9985_S1,2025-04-09,987.0,1.0,0.761464,0.307692


,unique_id,ds,y,promo_lag14,rel_price_lag14,promo_freq_26_lag14,department,class,subclass
185297,9985_S1,2025-03-26,1106.0,0.0,0.919012,0.307692,148,216,1021
185298,9985_S1,2025-04-02,1729.0,0.0,0.945223,0.307692,148,216,1021
185299,9985_S1,2025-04-09,987.0,1.0,0.761464,0.307692,148,216,1021


,unique_id,ds,promo_lag14,rel_price_lag14,promo_freq_26_lag14
93,10006_S1,2025-04-16,0.0,0.994759,0.115385
94,10006_S1,2025-04-23,0.0,0.997149,0.115385
95,10006_S1,2025-04-30,0.0,0.974827,0.115385


### 2. CatBoost

The attributes are listed twice. `static_features` tells MLForecast they are constant per series, so it repeats them for the forecast weeks and leaves them out of `X_df`. `cat_features` tells CatBoost to treat them as categories. It is a `fit` argument of CatBoost, so it goes through `models_fit_kwargs`, keyed by the model's name. Passing it to the `CatBoostRegressor` constructor fails: MLForecast copies each model with scikit-learn's `clone`, which can't reproduce that argument. CatBoost reads the string columns as categories as they are.

In [5]:
cb_categorical = MLForecast(
    models={"catboost_exog_categorical": CatBoostRegressor(**CB_PARAMS)},
    freq=FREQ,
    lags=LAGS,
    lag_transforms=LAG_TRANSFORMS,
    date_features=DATE_FEATURES,
    target_transforms=[LocalStandardScaler()],
    num_threads=1,
)

start = time.perf_counter()
cb_categorical.fit(
    with_categories,
    static_features=cat_cols,
    models_fit_kwargs={"catboost_exog_categorical": {"cat_features": cat_cols}},
)
cb_preds = cb_categorical.predict(H, X_df=future_exog)
cb_seconds = time.perf_counter() - start
cb_preds.head(3)

,unique_id,ds,catboost_exog_categorical
0,10006_S1,2025-04-16,1519.328220
1,10006_S1,2025-04-23,1564.947422
2,10006_S1,2025-04-30,1613.733408


### 3. LightGBM

The same model with LightGBM: same frames, same `static_features`, same `X_df`. LightGBM has no `cat_features` argument; it treats a column as categorical when its pandas dtype is `category`. So the attributes are converted first, and nothing goes in `models_fit_kwargs`.

In [6]:
with_category_dtype = with_categories.astype({col: "category" for col in cat_cols})
display(with_category_dtype[cat_cols].dtypes.to_frame("dtype").T)

lgbm_categorical = MLForecast(
    models={"lgbm_exog_categorical": lgb.LGBMRegressor(**LGB_PARAMS)},
    freq=FREQ,
    lags=LAGS,
    lag_transforms=LAG_TRANSFORMS,
    date_features=DATE_FEATURES,
    target_transforms=[LocalStandardScaler()],
    num_threads=1,
)

start = time.perf_counter()
lgbm_categorical.fit(with_category_dtype, static_features=cat_cols)
lgbm_preds = lgbm_categorical.predict(H, X_df=future_exog)
lgbm_seconds = time.perf_counter() - start
lgbm_preds.head(3)

,department,class,subclass
dtype,category,category,category


,unique_id,ds,lgbm_exog_categorical
0,10006_S1,2025-04-16,1656.071190
1,10006_S1,2025-04-23,1750.529892
2,10006_S1,2025-04-30,1909.324653


### Are the attributes treated as categories?

The features each fitted model sees, and how it treats them.

In [7]:
catboost_fit = cb_categorical.models_["catboost_exog_categorical"]
lgbm_fit = lgbm_categorical.models_["lgbm_exog_categorical"]
lgbm_infos = lgbm_fit.booster_.dump_model()["feature_infos"]
pd.DataFrame({
    "catboost": ["categorical" if i in catboost_fit.get_cat_feature_indices() else "numeric" for i in range(len(catboost_fit.feature_names_))],
    "lightgbm": ["categorical" if lgbm_infos[f]["values"] else "numeric" for f in catboost_fit.feature_names_],
}, index=catboost_fit.feature_names_)

,catboost,lightgbm
promo_lag14,numeric,numeric
rel_price_lag14,numeric,numeric
promo_freq_26_lag14,numeric,numeric
department,categorical,categorical
class,categorical,categorical
subclass,categorical,categorical
lag1,numeric,numeric
lag2,numeric,numeric
lag3,numeric,numeric
lag4,numeric,numeric


### 4. Every origin

Five models at the ten cutoffs:

| model | algorithm | training frame | `static_features` |
|---|---|---|---|
| `catboost_baseline` | CatBoost | sales only | none |
| `catboost_hist_exog` | CatBoost | + lagged drivers | `[]` |
| `catboost_exog_categorical` | CatBoost | + attributes as strings | the attributes, plus `cat_features` |
| `lgbm_hist_exog` | LightGBM | + lagged drivers | `[]` |
| `lgbm_exog_categorical` | LightGBM | + attributes as `category` | the attributes |

Each row of `experiments` is one model: its MLForecast object, which training frame it uses, and the arguments to `fit`. The two models fitted above are reused at the final cutoff instead of refit. `runtime_seconds` is fit plus forecast plus formatting, summed over the ten origins; a LightGBM baseline fit takes under a second (notebook 05).

In [8]:
settings = dict(freq=FREQ, lags=LAGS, lag_transforms=LAG_TRANSFORMS, date_features=DATE_FEATURES, num_threads=1)
cb_baseline = MLForecast(models={"catboost_baseline": CatBoostRegressor(**CB_PARAMS)}, target_transforms=[LocalStandardScaler()], **settings)
cb_hist_exog = MLForecast(models={"catboost_hist_exog": CatBoostRegressor(**CB_PARAMS)}, target_transforms=[LocalStandardScaler()], **settings)
lgbm_hist_exog = MLForecast(models={"lgbm_hist_exog": lgb.LGBMRegressor(**LGB_PARAMS)}, target_transforms=[LocalStandardScaler()], **settings)

#  name                          MLForecast        training frame          fit arguments
experiments = [
    ("catboost_baseline",          cb_baseline,      "history",              {}),
    ("catboost_hist_exog",         cb_hist_exog,     "with_exog",            dict(static_features=[])),
    ("catboost_exog_categorical",  cb_categorical,   "with_categories",      dict(static_features=cat_cols, models_fit_kwargs={"catboost_exog_categorical": {"cat_features": cat_cols}})),
    ("lgbm_hist_exog",             lgbm_hist_exog,   "with_exog",            dict(static_features=[])),
    ("lgbm_exog_categorical",      lgbm_categorical, "with_category_dtype",  dict(static_features=cat_cols)),
]

boost_forecasts = [to_long(cb_preds, TEST_CUTOFF, "03_catboost", "historical_drivers"),
                   to_long(lgbm_preds, TEST_CUTOFF, "03_catboost", "historical_drivers")]
runtime_seconds = {"catboost_exog_categorical": cb_seconds, "lgbm_exog_categorical": lgbm_seconds}

for cutoff in ALL_CUTOFFS:
    history = actuals[actuals.ds <= cutoff]
    frames = {"history": history}
    frames["with_exog"] = history.merge(hist_features, on=["unique_id", "ds"])
    frames["with_categories"] = frames["with_exog"].merge(attributes, on="unique_id")
    frames["with_category_dtype"] = frames["with_categories"].astype({col: "category" for col in cat_cols})
    future_exog = hist_features[(hist_features.ds > cutoff) & (hist_features.ds <= cutoff + pd.Timedelta(weeks=H))]

    for name, mlf, frame, fit_args in experiments:
        if cutoff == TEST_CUTOFF and name in ("catboost_exog_categorical", "lgbm_exog_categorical"):
            continue                                            # fitted in sections 2 and 3
        start = time.perf_counter()
        mlf.fit(frames[frame], **fit_args)
        preds = mlf.predict(H, X_df=None if frame == "history" else future_exog)
        information = "history" if frame == "history" else "historical_drivers"
        boost_forecasts.append(to_long(preds, cutoff, "03_catboost", information))
        runtime_seconds[name] = runtime_seconds.get(name, 0) + time.perf_counter() - start

boost_forecasts = pd.concat(boost_forecasts)

### 5. Results

With `lgbm_baseline` from notebook 02, the LightGBM model on sales only. Development origins, then the final 14 weeks (exploratory).

In [9]:
boost_candidates = pd.concat([load_forecasts(["02_ml_baseline"]).query("model == 'lgbm_baseline'"), boost_forecasts])
boost_columns = ["model", "information", "wape", "bias", "mean_rmsse", "median_rmsse", "apc", "coverage"]
display(pd.Series(runtime_seconds, name="runtime_seconds_all_origins").round(1).to_frame())
dev_board = leaderboard(boost_candidates, actuals, scope="dev")
dev_board[boost_columns].round(3)

,runtime_seconds_all_origins
catboost_exog_categorical,471.3
lgbm_exog_categorical,12.3
catboost_baseline,20.1
catboost_hist_exog,22.5
lgbm_hist_exog,7.0


,model,information,wape,bias,mean_rmsse,median_rmsse,apc,coverage
0,catboost_baseline,Sales history and calendar only,0.442,0.130,1.815,0.910,0.126,1.0
1,lgbm_baseline,Sales history and calendar only,0.449,0.139,1.865,0.921,0.136,1.0
2,lgbm_exog_categorical,Historical exogenous features,0.422,0.099,1.850,0.891,0.157,1.0
3,catboost_exog_categorical,Historical exogenous features,0.434,0.129,1.810,0.898,0.118,1.0
4,catboost_hist_exog,Historical exogenous features,0.438,0.131,1.818,0.908,0.118,1.0
5,lgbm_hist_exog,Historical exogenous features,0.444,0.141,1.877,0.922,0.136,1.0


In [10]:
test_board = leaderboard(boost_candidates, actuals, scope="test")
test_board[boost_columns].round(3)

,model,information,wape,bias,mean_rmsse,median_rmsse,apc,coverage
0,lgbm_baseline,Sales history and calendar only,0.419,0.109,1.302,0.733,0.101,1.0
1,catboost_baseline,Sales history and calendar only,0.424,0.113,1.312,0.746,0.092,1.0
2,lgbm_exog_categorical,Historical exogenous features,0.396,0.105,1.266,0.706,0.113,1.0
3,lgbm_hist_exog,Historical exogenous features,0.418,0.119,1.297,0.729,0.102,1.0
4,catboost_exog_categorical,Historical exogenous features,0.425,0.128,1.305,0.736,0.093,1.0
5,catboost_hist_exog,Historical exogenous features,0.427,0.126,1.307,0.741,0.092,1.0


WAPE by feature set and algorithm: down a column adds features, across a row changes the algorithm. Then each step as a change in WAPE points (negative is better).

In [11]:
dev_wape, test_wape = dev_board.set_index("model").wape, test_board.set_index("model").wape
grid = pd.DataFrame({
    "lightgbm": ["lgbm_baseline", "lgbm_hist_exog", "lgbm_exog_categorical"],
    "catboost": ["catboost_baseline", "catboost_hist_exog", "catboost_exog_categorical"],
}, index=["sales only", "+ lagged drivers", "+ categorical attributes"])
display(pd.concat({"development": grid.apply(lambda col: col.map(dev_wape)), "final 14 weeks": grid.apply(lambda col: col.map(test_wape))}, axis=1).round(3))

steps = pd.DataFrame([
    ("adding lagged drivers", "LightGBM", "lgbm_hist_exog", "lgbm_baseline"),
    ("adding lagged drivers", "CatBoost", "catboost_hist_exog", "catboost_baseline"),
    ("adding categorical attributes", "LightGBM", "lgbm_exog_categorical", "lgbm_hist_exog"),
    ("adding categorical attributes", "CatBoost", "catboost_exog_categorical", "catboost_hist_exog"),
    ("CatBoost instead of LightGBM", "sales only", "catboost_baseline", "lgbm_baseline"),
    ("CatBoost instead of LightGBM", "+ lagged drivers", "catboost_hist_exog", "lgbm_hist_exog"),
    ("CatBoost instead of LightGBM", "+ categorical attributes", "catboost_exog_categorical", "lgbm_exog_categorical"),
], columns=["change", "for", "new", "old"])
steps["development"] = (steps.new.map(dev_wape) - steps.old.map(dev_wape)) * 100
steps["final 14 weeks"] = (steps.new.map(test_wape) - steps.old.map(test_wape)) * 100
steps.set_index(["change", "for"])[["development", "final 14 weeks"]].round(2)

development          final 14 weeks         
                            lightgbm catboost       lightgbm catboost
sales only                     0.449    0.442          0.419    0.424
+ lagged drivers               0.444    0.438          0.418    0.427
+ categorical attributes       0.422    0.434          0.396    0.425

development  \
change                        for                                     
adding lagged drivers         LightGBM                        -0.57   
                              CatBoost                        -0.33   
adding categorical attributes LightGBM                        -2.19   
                              CatBoost                        -0.41   
CatBoost instead of LightGBM  sales only                      -0.77   
                              + lagged drivers                -0.53   
                              + categorical attributes         1.24   

                                                        final 14 weeks  
change                        for                                       
adding lagged drivers         LightGBM                           -0.07  
                              CatBoost                            0.30  
adding categorical attributes LightGBM                           -2.26  
                              CatBoost                           -0.17  
CatBoost instead of LightGBM  sales only                          0.52  
                              + lagged drivers                    0.90  
                              + categorical attributes            2.98

### Reading the result

- **Algorithm, same features.** CatBoost is slightly better than LightGBM on the development origins with the basic and lagged-driver features (0.442 against 0.449, 0.438 against 0.444), and slightly worse in the final window (0.424 against 0.419, 0.427 against 0.418). Neither difference holds across both periods.
- **Categorical attributes.** They help LightGBM much more than CatBoost: development WAPE 0.444 to 0.422 for LightGBM, 0.438 to 0.434 for CatBoost; final window 0.418 to 0.396 against 0.427 to 0.425. `lgbm_exog_categorical` is the best model in this table in both periods.
- **Bias and stability.** Every model here overforecasts (bias +0.10 to +0.14), as the L2 baseline does. CatBoost revises less between origins (development APC 0.118 to 0.126, against 0.136 to 0.157) and has a slightly lower development mean RMSSE.
- **Cost.** CatBoost takes about 2 seconds per fit and forecast without categorical features and about 41 seconds with them. LightGBM with the same categorical features takes about 1.3 seconds.

The gain from the hierarchy attributes is mostly a feature effect that LightGBM uses better here, not a reason to switch algorithm. With more historical drivers, `lgbm_hist_drivers` (notebook 03, section 7) does slightly better again: 0.416 on development and 0.396 in the final window.

Try it later: give the categorical models the L1 objective from notebook 06 (`loss_function="MAE"` for CatBoost, `objective="l1"` for LightGBM).

In [12]:
save_forecasts(boost_forecasts, "03_catboost")